# Data Processing Script

This script is used in Hew's replication of Dang et al. (2025) to process the raw data. CSV files from each participant were saved to my Google Drive via Data Pipe after the completion of the jsPsych study. The goal is to take each `.csv` file and convert it to the tidy/wide data format that we need. Each `.csv` that is outputted by jsPsych contains entries that are JSONS that we need to unravel to actually get the data that we need.

Claude code Sonnet 5.5 Medium (2026, Nov 17) was used to assist in coding this.

```
Before you begin, make sure that the file structure looks like:
.
├── Data
│   ├── 2324820432840.csv
│   └── 2482394023894.csv
└── JsonCleaning.ipynb
```

## Script to Extract and Pivot Data

In [ ]:
import pandas as pd
import json
from pathlib import Path # This is a built-in Python feature

# Path() creates an object representing a file directory to allow us to use object oriented methods
# to interact with the files. Path("Data") is selecting the "Data" folder that we have.
# glob() takes Unix style pathname pattern.

DataFiles = Path("Data").glob("*.csv")
LongRows = []

for file in DataFiles:
    df = pd.read_csv(file)
    # Because this column is uniform for each file, it does not matter which .iloc[] you choose as long as it is in range
    ID = df['subject_id'].iloc[0]

    # Grab the last time elpased (total time in Experiment)
    TotalTime = df['time_elapsed'].iloc[-1]
    TotalMinutes = (TotalTime/1000)/60

    LongRows.append({
        "ID":ID,
        "TrialIndex" : "X",
        "item" : "TotalMinutes",
        "value" : TotalMinutes
    })

    # Grab Condition
    Condition = df['condition'].iloc[0]
    LongRows.append({
        "ID":ID,
        "TrialIndex" :"X",
        "item" : 'Condition',
        "value" : Condition
            })
    for i, row in df.iterrows():
        # Keep looping through the responses until the response col 
        Response = row['response']
        if pd.isna(Response) or not str(Response).strip().startswith("{"):
            continue
        Parsed = json.loads(Response)
        if not isinstance(Parsed, dict):
            Parsed = {"response":Parsed}
        
        for key, value in Parsed.items():
            LongRows.append({
                "ID": ID,
                "TrialIndex": row["trial_index"],
                "item": key,
                "value": value,
            })
LongDataFrame = pd.DataFrame(LongRows)

# Create Temporary Column Names
LongDataFrame["Col"] = LongDataFrame["TrialIndex"].astype(str) + "_" + LongDataFrame["item"]


# Pivot the data so that each ID gets a row, each unique value in Col becomes a column, and use the "value" column in the dataframe to fill the cell.
WideDataFrame = LongDataFrame.pivot(index = "ID", columns = "Col", values = "value").reset_index()
WideDataFrame

## Fixing Likert-Scale Ranges
jsPsych outputs the position of the likert which is 0-indexed. My likerts ranged from 1–7. I need to add 1 to each to move the 0-6 range to 1-7.

In [ ]:
LikertColsToAdjust = ['10_Content', '10_Happy', '10_Joyful', '10_Pleased',
       '10_Satisfied','6_PretestMood', '8_Lonely1','8_Lonely2', '8_Lonely3', '9_Appreciative', '9_Grateful', '9_Thankful']
WideDataFrame[LikertColsToAdjust] +=1
WideDataFrame

## Drop Suspected AI Response

In [ ]:
WideDataFrame[WideDataFrame['11_AI_CAPTCHA'] == ""]

## Remove free responses from the data set for participant privacy.

In [ ]:
del WideDataFrame['7_GratitudeResponse']
del WideDataFrame['7_EventResponse']

## Format and Tidy Data Set

In [ ]:
# Remove Brackets from Gender and RacialEthnic columns
WideDataFrame['4_Gender'] = WideDataFrame['4_Gender'].astype(str).replace(r"[\[\]']", '', regex=True)
WideDataFrame['5_RacialEthnic'] = WideDataFrame['5_RacialEthnic'].astype(str).replace(r"[\[\]']", '', regex=True)
WideDataFrame

Rename columns

In [ ]:
WideDataFrame = WideDataFrame.rename(columns={
    '10_Content':'Content',
    '10_Happy' : 'Happy',
    '10_Joyful' : 'Joyful',
    '10_Pleased': 'Pleased',
    '10_Satisfied' : 'Satisfied',
    '11_AI_CAPTCHA':'AI_CAPTCHA',
    '3_Age' : 'Age',
    '4_Gender': 'Gender',
    '5_RacialEthnic': 'RacialEthnic',
    '6_PretestMood' : 'PretestMood',
    '8_Lonely1' : 'Lonely1',
    '8_Lonely2': 'Lonely2',
    '8_Lonely3' : 'Lonely3',
    '9_Appreciative':'Appreciative',
    '9_Grateful':'Grateful',
    '9_Thankful' : 'Thankful',
    'X_TotalMinutes' : 'TotalMinutesInStudy',
    'X_Condition':'Condition'
})

len(WideDataFrame.columns)

In [ ]:
WideDataFrame['Int_ID'] = range(1, len(WideDataFrame)+1)

Reorder columns chronologically.

In [ ]:
WideDataFrame = WideDataFrame[['Int_ID', 'Age', 'Gender', 'RacialEthnic', 'PretestMood','Condition', 'Lonely1','Lonely2', 'Lonely3', 'Appreciative', 'Grateful', 'Thankful', 'Content', 'Happy', 'Joyful', 'Pleased', 'Satisfied', 'AI_CAPTCHA','TotalMinutesInStudy']]

# Export the Data to CSV

In [ ]:
WideDataFrame.to_csv("CleanDataPublic.csv", index=False)
display(WideDataFrame)